# Credit Score & Loan Approval Modeling
This notebook builds **Linear Regression** (target: `credit_score`) and **Logistic Regression** (target: `approved_flag`, binary and multiclass) models using three model families:

1. Plain regression
2. Regularized regression (Ridge / Lasso / ElasticNet-style penalties)
3. Regression trained with different optimizers (Gradient Descent, Momentum, Adam, SAGA)

A single **generic experiment runner** fits/evaluates every combination and appends the results to `results/model_results.csv`, which can be used later for plotting/comparison.

## 1. Imports & Setup

In [2]:
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split, KFold, StratifiedKFold
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.linear_model import (
    LinearRegression, LogisticRegression,
)

from sklearn.metrics import (
    r2_score, mean_absolute_error, root_mean_squared_error,
    accuracy_score, f1_score, precision_score, recall_score,
    roc_auc_score, confusion_matrix
)

from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from scipy.stats import randint, uniform
from xgboost import XGBRegressor, XGBClassifier

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

In [3]:
RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)
RESULTS_CSV = RESULTS_DIR / "model_results.csv"
print(f"Results will be written to: {RESULTS_CSV.resolve()}")

Results will be written to: /home/neeraj/Projects/ml_course/results/model_results.csv


## 2. Load Data

In [4]:
df = pd.read_csv("data/cibil_score/cibil_score.csv")
df = df.drop(columns=["Unnamed: 0"])

# normalize column names
df.columns = [col.lower().strip() for col in df.columns]

print(df.shape)
print("Duplicate rows:", df.duplicated().sum())
df = df.drop_duplicates().reset_index(drop=True)
df.head()

(51336, 87)
Duplicate rows: 0


,prospectid,total_tl,tot_closed_tl,tot_active_tl,total_tl_opened_l6m,tot_tl_closed_l6m,pct_tl_open_l6m,pct_tl_closed_l6m,pct_active_tl,pct_closed_tl,...,pct_cc_enq_l6m_of_l12m,pct_pl_enq_l6m_of_ever,pct_cc_enq_l6m_of_ever,max_unsec_exposure_inpct,hl_flag,gl_flag,last_prod_enq2,first_prod_enq2,credit_score,approved_flag
0,1,5,4,1,0,0,0.000,0.0,0.200,0.800,...,0.0,0.0,0.0,13.333,1,0,PL,PL,696,P2
1,2,1,0,1,0,0,0.000,0.0,1.000,0.000,...,0.0,0.0,0.0,0.860,0,0,ConsumerLoan,ConsumerLoan,685,P2
2,3,8,0,8,1,0,0.125,0.0,1.000,0.000,...,0.0,0.0,0.0,5741.667,1,0,ConsumerLoan,others,693,P2
3,4,1,0,1,1,0,1.000,0.0,1.000,0.000,...,0.0,0.0,0.0,9.900,0,0,others,others,673,P2
4,5,3,2,1,0,0,0.000,0.0,0.333,0.667,...,0.0,0.0,0.0,-99999.000,0,0,AL,AL,753,P1


In [5]:
df["approved_flag"].value_counts()

approved_flag
P2    32199
P3     7452
P4     5882
P1     5803
Name: count, dtype: int64

## 3. Feature / Target Setup
* `X` -> all columns except `credit_score` and `approved_flag`
* `y_linear` -> `credit_score` (Linear Regression target)
* `y_multiclass` -> `approved_flag` (Logistic Regression, multiclass: P1/P2/P3/P4)
* `y_binary` -> `approved_flag` mapped to **1** for P1/P2 and **0** for P3/P4 (Logistic Regression, binary)

In [6]:
X = df.drop(columns=["approved_flag", "credit_score"])
y_linear = df["credit_score"].astype(float)
y_multiclass = df["approved_flag"].astype(str)

binary_map = {"P1": 1, "P2": 1, "P3": 0, "P4": 0}
y_binary = df["approved_flag"].map(binary_map)

assert y_binary.isna().sum() == 0, "approved_flag has values outside P1-P4"

print(y_multiclass.value_counts())
print(y_binary.value_counts())

approved_flag
P2    32199
P3     7452
P4     5882
P1     5803
Name: count, dtype: int64
approved_flag
1    38002
0    13334
Name: count, dtype: int64


## 4. Train / Test Split
A single split on `X` is created and then reused (via the shared index) for every target so that every model family sees the exact same rows in train/test.

In [7]:
X_train, X_test, idx_train, idx_test = train_test_split(
    X, X.index, test_size=0.2, random_state=RANDOM_STATE
)

y_linear_train, y_linear_test = y_linear.loc[idx_train], y_linear.loc[idx_test]
y_multi_train, y_multi_test = y_multiclass.loc[idx_train], y_multiclass.loc[idx_test]
y_bin_train, y_bin_test = y_binary.loc[idx_train], y_binary.loc[idx_test]

print(X_train.shape, X_test.shape)

(41068, 85) (10268, 85)


## 5. Missing Value Handling
Domain-specific cleanup carried over/expanded from the original notebook. The dataset encodes several kinds of \"missing\" as the sentinel value `-99999`; the correct treatment depends on what the field means.

In [8]:
# 5a. Drop columns with very high missingness
high_missing = [c for c in ["cc_utilization", "pl_utilization"] if c in X_train.columns]
X_train = X_train.drop(columns=high_missing)
X_test = X_test.drop(columns=high_missing)

# 5b. Median-impute a handful of numeric fields where -99999 means "unknown"
median_columns = [
    "age_oldest_tl", "age_newest_tl", "pct_currentbal_all_tl", "time_since_recent_payment",
]
for col in median_columns:
    if col not in X_train.columns:
        continue
    X_train[col] = X_train[col].replace(-99999, np.nan)
    X_test[col] = X_test[col].replace(-99999, np.nan)
    train_median = X_train[col].median()
    X_train[col] = X_train[col].fillna(train_median)
    X_test[col] = X_test[col].fillna(train_median)  # use TRAIN median to avoid leakage

# 5c. Delinquency fields: -99999 means "never delinquent" -> 0
delinquency_columns = [
    "max_delinquency_level", "max_deliq_6mts", "max_deliq_12mts",
    "time_since_recent_deliquency", "time_since_first_deliquency",
]
delinquency_columns = [c for c in delinquency_columns if c in X_train.columns]
X_train[delinquency_columns] = X_train[delinquency_columns].replace(-99999, 0)
X_test[delinquency_columns] = X_test[delinquency_columns].replace(-99999, 0)

# 5d. Enquiry fields: -99999 means "no enquiry" -> 0
enquiry_columns = [
    "tot_enq", "cc_enq", "pl_enq", "cc_enq_l6m", "cc_enq_l12m",
    "pl_enq_l6m", "pl_enq_l12m", "enq_l3m", "enq_l6m", "enq_l12m",
]
enquiry_columns = [c for c in enquiry_columns if c in X_train.columns]
X_train[enquiry_columns] = X_train[enquiry_columns].replace(-99999, 0)
X_test[enquiry_columns] = X_test[enquiry_columns].replace(-99999, 0)

# 5e. time_since_recent_enq: -99999 = "no enquiry ever" -> longer than any observed gap,
# keep the "no enquiry" signal as its own binary flag (fit on TRAIN only, applied to both)
col = "time_since_recent_enq"
if col in X_train.columns:
    train_mask = X_train[col] == -99999
    test_mask = X_test[col] == -99999
    fill_value = X_train.loc[~train_mask, col].max() + 1

    X_train["no_enquiry_flag"] = train_mask.astype(int)
    X_test["no_enquiry_flag"] = test_mask.astype(int)

    X_train[col] = X_train[col].replace(-99999, fill_value)
    X_test[col] = X_test[col].replace(-99999, fill_value)

# 5f. max_unsec_exposure_inpct: -99999 = "no unsecured loan" -> 0%
col = "max_unsec_exposure_inpct"
if col in X_train.columns:
    X_train[col] = X_train[col].replace(-99999, 0)
    X_test[col] = X_test[col].replace(-99999, 0)

# 5g. Log-transform skewed income
if "netmonthlyincome" in X_train.columns:
    X_train["netmonthlyincome_log"] = np.log1p(X_train["netmonthlyincome"].clip(lower=0))
    X_test["netmonthlyincome_log"] = np.log1p(X_test["netmonthlyincome"].clip(lower=0))
    X_train = X_train.drop(columns=["netmonthlyincome"])
    X_test = X_test.drop(columns=["netmonthlyincome"])

print("Remaining NaNs in X_train:", X_train.isna().sum().sum())
print("Remaining NaNs in X_test:", X_test.isna().sum().sum())

Remaining NaNs in X_train: 0
Remaining NaNs in X_test: 0


## 6. Preprocessing Pipeline (scale numeric, one-hot encode categorical)

In [9]:
numeric_columns = X_train.select_dtypes(include=np.number).columns
categorical_columns = X_train.select_dtypes(include="object").columns
print(f"{len(numeric_columns)} numeric columns, {len(categorical_columns)} categorical columns")

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_columns),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_columns),
    ]
)

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

feature_names = preprocessor.get_feature_names_out()
X_train_processed = pd.DataFrame(X_train_processed, columns=feature_names, index=X_train.index)
X_test_processed = pd.DataFrame(X_test_processed, columns=feature_names, index=X_test.index)

print("Processed shapes:", X_train_processed.shape, X_test_processed.shape)

79 numeric columns, 5 categorical columns
Processed shapes: (41068, 102) (10268, 102)


## 7. Hyperparameter Search

Grid Search is being executed for three model families: LinearRegression, RandomForestRegressor, XGBRegressor


In [10]:
ALL_RESULTS = []

def evaluate_regression(model, X_te, y_te):
    y_pred = model.predict(X_te)
    return {
        "r2": r2_score(y_te, y_pred),
        "mae": mean_absolute_error(y_te, y_pred),
        "rmse": root_mean_squared_error(y_te, y_pred),
    }

def evaluate_classification(model, X_te, y_te, binary):
    y_pred = model.predict(X_te)
    metrics = {
        "accuracy": accuracy_score(y_te, y_pred),
        "f1_macro": f1_score(y_te, y_pred, average="macro", zero_division=0),
        "precision_macro": precision_score(y_te, y_pred, average="macro", zero_division=0),
        "recall_macro": recall_score(y_te, y_pred, average="macro", zero_division=0),
        # Weighted averages
        "precision_weighted": precision_score(
            y_te, y_pred, average="weighted", zero_division=0
        ),
        "recall_weighted": recall_score(
            y_te, y_pred, average="weighted", zero_division=0
        ),
        "f1_weighted": f1_score(
            y_te, y_pred, average="weighted", zero_division=0
        ),
        # Confusion Matrix
        "confusion_matrix": confusion_matrix(y_te, y_pred)        
    }
    if binary and hasattr(model, "predict_proba"):
        y_prob = model.predict_proba(X_te)[:, 1]
        metrics["roc_auc"] = roc_auc_score(y_te, y_prob)
    return metrics

In [11]:
# Initializating kfold and skfold to 3.

HPARAM_RESULTS = []

# Lighter, 3-fold CV specifically for hyperparameter search (Section 14's 5-fold kfold/skfold
# are reused everywhere else; tuning many models x many candidates is far more expensive, so a
# smaller cv here keeps runtime reasonable). Bump back to 5 if you have the compute budget.
search_kfold = KFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
search_skfold = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
RANDOM_SEARCH_N_ITER = 5  # number of sampled combinations for RandomizedSearchCV

# XGBoost needs 0..n_classes-1 integer labels for multiclass
label_encoder = LabelEncoder()
y_multi_train_enc = label_encoder.fit_transform(y_multi_train)
y_multi_test_enc = label_encoder.transform(y_multi_test)
print("Multiclass label mapping:", dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_))))

Multiclass label mapping: {'P1': 0, 'P2': 1, 'P3': 2, 'P4': 3}


In [19]:
def linear_model_specs():
    """model_name -> (estimator, grid_search_param_grid, random_search_param_distributions)"""
    return {
        "linear_regression": (
            LinearRegression(),
            {"fit_intercept": [True, False], "positive": [True, False]},
            {"fit_intercept": [True, False], "positive": [True, False]},
        ),
        "random_forest": (
            RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=1),
            {
                "n_estimators": [100, 150],
                "max_depth": [5, 10],
            },
            {
                "n_estimators": randint(50, 250),
                "max_depth": [3, 5, 8, None],
                "min_samples_leaf": randint(1, 6),
            },
        ),
        "xgboost": (
            XGBRegressor(random_state=RANDOM_STATE, verbosity=0, n_jobs=1),
            {
                "n_estimators": [100, 150],
                "max_depth": [3, 5],
            },
            {
                "n_estimators": randint(50, 250),
                "max_depth": randint(2, 8),
                "learning_rate": uniform(0.03, 0.27),
                "subsample": uniform(0.6, 0.4),
            },
        ),
    }


def classification_model_specs():
    """model_name -> (estimator, grid_search_param_grid, random_search_param_distributions)"""
    return {
        "logistic_regression": (
            LogisticRegression(solver="saga", max_iter=2000, random_state=RANDOM_STATE),
            {"C": [0.1, 1.0, 10.0], "l1_ratio": [0.0, 0.5, 1.0]},
            {"C": uniform(0.01, 15), "l1_ratio": uniform(0, 1)},
        ),
        "random_forest": (
            RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=1),
            {
                "n_estimators": [200, 300],
                "min_samples_leaf": [1, 2, 5],
                "class_weight": [
                    {
                        "P1": 1.0,
                        "P2": 0.5,
                        "P3": 2.0,
                        "P4": 1.0
                    },
                    {
                        "P1": 1.0,
                        "P2": 0.75,
                        "P3": 1.75,
                        "P4": 1.0
                    },
                    {
                        "P1": 1.0,
                        "P2": 0.85,
                        "P3": 2.0,
                        "P4": 1.0
                    },
                ],
                "max_depth": [5, 8],
                "max_features": ["sqrt", 0.7]
            },
            {
                "n_estimators": randint(50, 250),
                "max_depth": [3, 5, 8, None],
                "min_samples_leaf": randint(1, 6),
            },
        ),
        "xgboost": (
            XGBClassifier(random_state=RANDOM_STATE, verbosity=0, eval_metric="logloss", n_jobs=1),
            {
                "n_estimators": [100, 150],
                "max_depth": [3, 5],
            },
            {
                "n_estimators": randint(50, 250),
                "max_depth": randint(2, 8),
                "learning_rate": uniform(0.03, 0.27),
                "subsample": uniform(0.6, 0.4),
            },
        ),
    }

### GridSearchCV for Linear Regression

In [16]:
linear_model_specs().items()

dict_items([('linear_regression', (LinearRegression(), {'fit_intercept': [True, False], 'positive': [True, False]}, {'fit_intercept': [True, False], 'positive': [True, False]})), ('random_forest', (RandomForestRegressor(n_jobs=1, random_state=42), {'n_estimators': [100, 150], 'max_depth': [5, 10]}, {'n_estimators': <scipy.stats._distn_infrastructure.rv_discrete_frozen object at 0x711a027202c0>, 'max_depth': [3, 5, 8, None], 'min_samples_leaf': <scipy.stats._distn_infrastructure.rv_discrete_frozen object at 0x711a027200e0>})), ('xgboost', (XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=True, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=None, max_bin=None, max_c

In [ ]:
param_grid_linreg = linear_model_specs().get("random_forest")[1]
# X_train_processed, y_linear_train, X_test_processed, y_linear_test,
cv=search_kfold
scoring="r2"
grid_search_linreg = GridSearchCV(LinearRegression(), param_grid_linreg, cv=cv, scoring=scoring, n_jobs=-1)
start = time.time()
grid_search_linreg.fit(X_train_processed, y_linear_train)
search_time = time.time() - start
best_model = grid_search_linreg.best_estimator_
result = evaluate_regression(best_model, X_test_processed, y_linear_test)
print(f"Time Taken: {search_time}")
print("Best Parameters:", grid_search_linreg.best_params_)
print("Best CV R²:", grid_search_linreg.best_score_)
print(f"Result: {result}")

Time Taken: 35.23580074310303
Best Parameters: {'fit_intercept': True, 'positive': False}
Best CV R²: 0.91208216543814
Result: {'r2': 0.910588710512521, 'mae': 5.318176216703804, 'rmse': 6.115563946889853}


### GridSearchCV for Random Forest Regression

In [16]:
numeric_columns = X_train.select_dtypes(include=np.number).columns
categorical_columns = X_train.select_dtypes(include="object").columns
print(f"{len(numeric_columns)} numeric columns, {len(categorical_columns)} categorical columns")

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_columns),
    ]
)

X_train_proc_rf = preprocessor.fit_transform(X_train)
X_test_proc_rf = preprocessor.transform(X_test)

feature_names = preprocessor.get_feature_names_out()
X_train_proc_rf = pd.DataFrame(X_train_proc_rf, columns=feature_names, index=X_train.index)
X_test_proc_rf = pd.DataFrame(X_test_proc_rf, columns=feature_names, index=X_test.index)

79 numeric columns, 5 categorical columns


In [23]:
cv = search_skfold
scoring = "f1_macro"
param_grid_rfreg = classification_model_specs().get("random_forest")
estimator = param_grid_rfreg[0]
param_grid = param_grid_rfreg[1]
rf_multiclass_grid_search = GridSearchCV(estimator, param_grid, cv=cv, scoring=scoring, n_jobs=-1)
start = time.time()
rf_multiclass_grid_search.fit(X_train_proc_rf, y_multi_train)
search_time = time.time() - start
best_model = rf_multiclass_grid_search.best_estimator_
results = evaluate_classification(rf_multiclass_grid_search, X_test_proc_rf, y_multi_test, False)
print(f"Time Taken: {search_time}")
print("Best Parameters:", rf_multiclass_grid_search.best_params_)
print("Best CV F1 Macro:", rf_multiclass_grid_search.best_score_)
print(f"Result: {results}")

Time Taken: 638.1898288726807
Best Parameters: {'class_weight': {'P1': 1.0, 'P2': 0.5, 'P3': 2.0, 'P4': 1.0}, 'max_depth': 8, 'max_features': 0.7, 'min_samples_leaf': 2, 'n_estimators': 300}
Best CV F1 Macro: 0.2510252693167206
Result: {'accuracy': 0.4383521620568757, 'f1_macro': 0.25046624957931046, 'precision_macro': 0.3641965482800113, 'recall_macro': 0.3129095213528415, 'precision_weighted': 0.5288586285917384, 'recall_weighted': 0.4383521620568757, 'f1_weighted': 0.4331801685791258, 'confusion_matrix': array([[  53,  722,  358,    2],
       [  87, 3423, 2857,   12],
       [  16,  496, 1019,    5],
       [  10,  239,  963,    6]])}


In [28]:
best_model.feature_importances_
feature_names = preprocessor.get_feature_names_out()
feature_names
importance_df = pd.DataFrame({
    "feature": feature_names,
    "importance": best_model.feature_importances_
}).sort_values(
    "importance",
    ascending=False
)

print(importance_df.head(30))

                              feature  importance
16         cat__last_prod_enq2_others    0.309850
13   cat__last_prod_enq2_ConsumerLoan    0.097377
22        cat__first_prod_enq2_others    0.082944
1           cat__maritalstatus_Single    0.063425
0          cat__maritalstatus_Married    0.056608
15             cat__last_prod_enq2_PL    0.055730
19  cat__first_prod_enq2_ConsumerLoan    0.047747
21            cat__first_prod_enq2_PL    0.034933
11             cat__last_prod_enq2_AL    0.024394
20            cat__first_prod_enq2_HL    0.023950
14             cat__last_prod_enq2_HL    0.022832
3             cat__education_GRADUATE    0.022428
2                 cat__education_12TH    0.020092
10                      cat__gender_M    0.016862
9                       cat__gender_F    0.016839
17            cat__first_prod_enq2_AL    0.016339
7                  cat__education_SSC    0.015466
5        cat__education_POST-GRADUATE    0.015101
12             cat__last_prod_enq2_CC    0.014888


In [21]:
classification_model_specs().get("random_forest")

(RandomForestClassifier(n_jobs=1, random_state=42),
 {'n_estimators': [200, 300],
  'min_samples_leaf': [1, 2, 5],
  'class_weight': [{'P1': 1.0, 'P2': 0.5, 'P3': 2.0, 'P4': 1.0},
   {'P1': 1.0, 'P2': 0.75, 'P3': 1.75, 'P4': 1.0},
   {'P1': 1.0, 'P2': 0.85, 'P3': 2.0, 'P4': 1.0}],
  'max_depth': [5, 8],
  'max_features': ['sqrt', 0.7]},
 {'n_estimators': <scipy.stats._distn_infrastructure.rv_discrete_frozen at 0x7d2ed941fe30>,
  'max_depth': [3, 5, 8, None],
  'min_samples_leaf': <scipy.stats._distn_infrastructure.rv_discrete_frozen at 0x7d2ed49b73e0>})

### GridSearchCV for XGBoost Regression

### RandomSearchCV for Random Forest Regression

### RandomSearchCV for XGBoost Regression

### GridSearchSF for Logistic Regression binary classification

How to choose CV vs SF?

If case of imbalance among target variables, we use Startified fold else, use kfold

As we have imbalance for P1 class so will use SF


In [ ]:
cv = search_skfold
scoring = "f1_macro"
params = classification_model_specs().get("logistic_regression")
estimator = params[0] # LogisticRegression(solver="saga", max_iter=2000, random_state=RANDOM_STATE)
param_grid = params[2]
logreg_bin_grid_search = GridSearchCV(estimator, param_grid, cv=cv, scoring=scoring, n_jobs=-1)
start = time.time()
logreg_bin_grid_search.fit(X_train_processed, y_bin_train)
search_time = time.time() - start
best_model = logreg_bin_grid_search.best_estimator_
results = evaluate_classification(logreg_bin_grid_search, X_test_processed, y_bin_test, True)
print(f"Time Taken: {search_time}")
print("Best Parameters:", logreg_bin_grid_search.best_params_)
print("Best CV F1 Macro:", logreg_bin_grid_search.best_score_)
print(f"Result: {results}")

Time Taken: 1195.9758958816528
Best Parameters: {'C': 1.0, 'l1_ratio': 0.5}
Best CV R²: 0.8536411855767966
Result: {'accuracy': 0.8931632255551227, 'f1_macro': 0.8573097881732397, 'precision_macro': 0.8780544803840213, 'recall_macro': 0.8416642842825081, 'precision_weighted': 0.891050076158225, 'recall_weighted': 0.8931632255551227, 'f1_weighted': 0.8904674260084374, 'confusion_matrix': array([[2012,  742],
       [ 355, 7159]]), 'roc_auc': 0.9353353768680455}


### RandomSearchSF for Logistic Regression binary classification

In [29]:
cv = search_skfold
scoring = "f1_macro"
params = classification_model_specs().get("logistic_regression")
estimator = params[0] # LogisticRegression(solver="saga", max_iter=2000, random_state=RANDOM_STATE)
param_distributions = params[1]
logreg_bin_random_search = RandomizedSearchCV(
            estimator, param_distributions, n_iter=RANDOM_SEARCH_N_ITER, cv=cv, scoring=scoring,
            random_state=RANDOM_STATE, n_jobs=-1)
start = time.time()
logreg_bin_random_search.fit(X_train_processed, y_bin_train)
search_time = time.time() - start
best_model = logreg_bin_random_search.best_estimator_
results = evaluate_classification(logreg_bin_random_search, X_test_processed, y_bin_test, True)
print(f"Time Taken: {search_time}")
print("Best Parameters:", logreg_bin_random_search.best_params_)
print("Best CV F1 Macro:", logreg_bin_random_search.best_score_)
print(f"Result: {results}")

Time Taken: 1522.8159019947052
Best Parameters: {'l1_ratio': 0.5, 'C': 1.0}
Best CV F1 Macro: 0.8536411855767966
Result: {'accuracy': 0.8931632255551227, 'f1_macro': 0.8573097881732397, 'precision_macro': 0.8780544803840213, 'recall_macro': 0.8416642842825081, 'precision_weighted': 0.891050076158225, 'recall_weighted': 0.8931632255551227, 'f1_weighted': 0.8904674260084374, 'confusion_matrix': array([[2012,  742],
       [ 355, 7159]]), 'roc_auc': 0.9353353768680455}


### RandomSearchSF for Logistic Regression multi classification

In [31]:
cv = search_skfold
scoring = "f1_macro"
params = classification_model_specs().get("logistic_regression")
estimator = params[0] # LogisticRegression(solver="saga", max_iter=2000, random_state=RANDOM_STATE)
param_distributions = params[1]
logreg_multi_random_search = RandomizedSearchCV(
            estimator, param_distributions, n_iter=RANDOM_SEARCH_N_ITER, cv=cv, scoring=scoring,
            random_state=RANDOM_STATE, n_jobs=-1)
start = time.time()
logreg_multi_random_search.fit(X_train_processed, y_multi_train_enc)
search_time = time.time() - start
best_model = logreg_multi_random_search.best_estimator_
results = evaluate_classification(logreg_multi_random_search, X_test_processed, y_multi_test_enc, False)
print(f"Time Taken: {search_time}")
print("Best Parameters:", logreg_multi_random_search.best_params_)
print("Best CV F1 Macro:", logreg_multi_random_search.best_score_)
print(f"Result: {results}")

Time Taken: 2114.987954854965
Best Parameters: {'l1_ratio': 1.0, 'C': 10.0}
Best CV F1 Macro: 0.708228720901004
Result: {'accuracy': 0.802493182703545, 'f1_macro': 0.7058207039932964, 'precision_macro': 0.7353354018966146, 'recall_macro': 0.6905239765198988, 'precision_weighted': 0.7793466830013531, 'recall_weighted': 0.802493182703545, 'f1_weighted': 0.7854280525925812, 'confusion_matrix': array([[ 891,  244,    0,    0],
       [ 130, 5999,  241,    9],
       [  26,  843,  422,  245],
       [   0,   43,  247,  928]])}


#### Pick wisely as above Logistic Grid search took 19 mins.

### GridSearchCV for Random Forest binary classification - can be removed as classes imbalanced SF makes more sense
### GridSearchCV for XGBoost binary classification - just for comparison can keep
### GridSearchSF for Random Forest binary classification - can be removed
### GridSearchSF for XGBoost binary classification - just for comparison can keep

### RandomSearchCV for Random Forest binary classification - can be removed as classes imbalanced SF makes more sense
### RandomSearchCV for XGBoost binary classification - just for comparison can keep
### RandomSearchSF for Random Forest binary classification
### RandomSearchSF for XGBoost binary classification - just for comparison can keep
### RandomSearchCV for Random Forest multi classification 
### RandomSearchCV for XGBoost multi classification
### RandomSearchSF for Random Forest multi classification
### RandomSearchSF for XGBoost multi classification

In [ ]:
def run_hyperparam_search(task, model_name, estimator, param_grid, param_distributions,
                           X_tr, y_tr, X_te, y_te, cv, scoring, n_iter=RANDOM_SEARCH_N_ITER,
                           results_store=HPARAM_RESULTS):
    """Run GridSearchCV then RandomizedSearchCV for one estimator, evaluate the best
    estimator from each on the held-out test set, and append one row per search type."""
    searches = {
        "grid_search": GridSearchCV(estimator, param_grid, cv=cv, scoring=scoring, n_jobs=-1),
        "random_search": RandomizedSearchCV(
            estimator, param_distributions, n_iter=n_iter, cv=cv, scoring=scoring,
            random_state=RANDOM_STATE, n_jobs=-1,
        ),
    }

    rows = []
    for search_type, search in searches.items():
        start = time.time()
        search.fit(X_tr, y_tr)
        search_time = time.time() - start

        best_model = search.best_estimator_
        row = {
            "task": task,
            "model": model_name,
            "search_type": search_type,
            "best_params": str(search.best_params_),
            "best_cv_score": search.best_score_,
            "search_time_sec": round(search_time, 2),
        }
        if task == "linear":
            row.update(evaluate_regression(best_model, X_te, y_te))
        else:
            row.update(evaluate_classification(best_model, X_te, y_te, binary=(task == "logistic_binary")))

        results_store.append(row)
        rows.append(row)
        print(f"[{task:>18}] [{model_name:>17}] {search_type:<14} "
              f"best_cv_score={search.best_score_:.4f} time={search_time:.1f}s "
              f"params={search.best_params_}")
    return rows

In [ ]:
# --- Linear regression target: credit_score ---
for model_name, (estimator, grid, dist) in linear_model_specs().items():
    run_hyperparam_search(
        "linear", model_name, estimator, grid, dist,
        X_train_processed, y_linear_train, X_test_processed, y_linear_test,
        cv=search_kfold, scoring="r2",
    )

In [ ]:
# --- Logistic regression target: approved_flag, BINARY (P1/P2 -> 1, P3/P4 -> 0) ---
for model_name, (estimator, grid, dist) in classification_model_specs().items():
    run_hyperparam_search(
        "logistic_binary", model_name, estimator, grid, dist,
        X_train_processed, y_bin_train, X_test_processed, y_bin_test,
        cv=search_skfold, scoring="f1_macro",
    )

In [ ]:
# --- Logistic regression target: approved_flag, MULTICLASS (P1/P2/P3/P4) ---
# XGBoost gets the label-encoded target; everything else keeps the original string labels.
for model_name, (estimator, grid, dist) in classification_model_specs().items():
    if model_name == "xgboost":
        run_hyperparam_search(
            "logistic_multiclass", model_name, estimator, grid, dist,
            X_train_processed, y_multi_train_enc, X_test_processed, y_multi_test_enc,
            cv=search_skfold, scoring="f1_macro",
        )
    else:
        run_hyperparam_search(
            "logistic_multiclass", model_name, estimator, grid, dist,
            X_train_processed, y_multi_train, X_test_processed, y_multi_test,
            cv=search_skfold, scoring="f1_macro",
        )

In [ ]:
hparam_results_df = pd.DataFrame(HPARAM_RESULTS)
HPARAM_RESULTS_CSV = RESULTS_DIR / "hyperparam_search_results.csv"
hparam_results_df.to_csv(HPARAM_RESULTS_CSV, index=False)
print(f"Saved {len(hparam_results_df)} hyperparameter-search rows to {HPARAM_RESULTS_CSV}")
hparam_results_df.sort_values(["task", "model", "search_type"])

In [ ]:
# Grid search vs. random search: best CV score per (task, model)
pivot = hparam_results_df.pivot_table(
    index=["task", "model"], columns="search_type", values="best_cv_score"
)
pivot